# Patent Topic Modeling with LDA

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import tomotopy as tp
from collections import Counter
# specify which corpus version to load (used to build the file path below)
version_prefix = "v6"

## Data Loading and Preprocessing

In [2]:


# load data for the chosen prefix
base_data = Path("../../data/claims_added")
data_path = base_data / f"{version_prefix}_processed.csv"

df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape[0]} patents")
df["text"] = df["claims"]  # downstream code uses this name

# Quick tensor check
tensor_count = df["text"].str.contains("tensor", case=False, na=False).sum()
print(f"Patents containing 'tensor': {tensor_count}")

# load custom stopwords from file
stopfile = Path("custom_stopwords.txt")
if not stopfile.exists():
    raise FileNotFoundError(f"custom stopwords file not found: {stopfile}")

with open(stopfile, "r") as f:
    custom_stopwords = [line.strip() for line in f if line.strip()]

# combine English stopwords with custom stopwords
stop_words = set(ENGLISH_STOP_WORDS)
for w in custom_stopwords:
    for part in w.split():
        stop_words.add(part.lower())

# text cleaning utility
def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)   # keep alnum/_/-
    s = re.sub(r"\b\d{1,2}\b", " ", s)      # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_unigrams(s: str) -> list[str]:
    s = clean_text(s)
    tokens = re.findall(r"\b[a-zA-Z][a-zA-Z0-9_]{1,}\b", s)
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

def build_bigram_vocab(token_lists: list[list[str]], min_count: int = 100) -> set[tuple[str, str]]:
    """
    Build a vocabulary of frequent adjacent bigrams across the corpus.
    """
    bigram_counts = Counter()

    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            bigram_counts[(tokens[i], tokens[i + 1])] += 1

    return {bg for bg, count in bigram_counts.items() if count >= min_count}

def add_bigrams(tokens: list[str], bigram_vocab: set[tuple[str, str]]) -> list[str]:
    """
    Keep unigrams and also append frequent bigrams as underscore-joined tokens.
    Example:
      ['tensor', 'core', 'unit'] ->
      ['tensor', 'core', 'unit', 'tensor_core', 'core_unit']  # if both are in vocab
    """
    out = list(tokens)

    for i in range(len(tokens) - 1):
        bg = (tokens[i], tokens[i + 1])
        if bg in bigram_vocab:
            out.append(tokens[i] + "_" + tokens[i + 1])

    return out

# Step 1: cleaned text
df["text_clean"] = df["text"].map(clean_text)

# Step 2: unigram tokenization
df["tokens_unigram"] = df["text"].map(tokenize_unigrams)

# drop docs that are already empty
df = df[df["tokens_unigram"].map(len) > 0].copy()

# Step 3: build corpus-wide bigram vocab
bigram_vocab = build_bigram_vocab(df["tokens_unigram"].tolist(), min_count=100)

print(f"Frequent bigrams kept: {len(bigram_vocab)}")
print("Sample bigrams:", [f"{a}_{b}" for a, b in list(sorted(bigram_vocab))[:20]])

# Step 4: add bigrams into each document
df["tokens"] = df["tokens_unigram"].map(lambda toks: add_bigrams(toks, bigram_vocab))

# final drop of any empty docs, just in case
df = df[df["tokens"].map(len) > 0].copy()

print(f"Usable documents: {len(df)}")
print(f"Average unigram tokens per doc: {df['tokens_unigram'].map(len).mean():.1f}")
print(f"Average tokens per doc after bigrams: {df['tokens'].map(len).mean():.1f}")

# quick sanity check
for i in range(min(3, len(df))):
    print(f"\nDoc {i} sample tokens:")
    print(df.iloc[i]["tokens"][:40])

Dataset loaded: 29481 patents
Patents containing 'tensor': 820
Frequent bigrams kept: 20237
Sample bigrams: ['abs_abs', 'absolute_difference', 'absolute_differences', 'absolute_value', 'absolute_values', 'abstraction_layer', 'ac_power', 'accelerated_data', 'accelerated_processing', 'acceleration_component', 'acceleration_data', 'acceleration_device', 'acceleration_devices', 'acceleration_hardware', 'acceleration_method', 'acceleration_structure', 'accelerator_accelerator', 'accelerator_according', 'accelerator_apparatus', 'accelerator_card']
Usable documents: 29473
Average unigram tokens per doc: 612.7
Average tokens per doc after bigrams: 845.5

Doc 0 sample tokens:
['method', 'host', 'implementing', 'distributed', 'logical', 'router', 'logical', 'switches', 'logical', 'network', 'hosts', 'message', 'data', 'compute', 'node', 'dcn', 'executing', 'host', 'logically', 'forwarding', 'message', 'distributed', 'logical', 'router', 'uses', 'particular', 'anycast', 'internet', 'protocol', 'i

## LDA Model Fitting

In [3]:
n_topics = 30

mdl = tp.LDAModel(
    k=n_topics,
    alpha=0.3,   # like doc_topic_prior
    eta=0.01,    # like topic_word_prior
    min_df=10,   # similar spirit to CountVectorizer(min_df=10)
    rm_top=0,    # can raise later if generic patent terms dominate
    seed=42
)

for tokens in df["tokens"]:
    mdl.add_doc(tokens)

print(f"Documents in model: {len(mdl.docs)}")
print(f"Vocabulary size: {mdl.num_vocabs}")

Documents in model: 29473
Vocabulary size: 0


In [4]:
print("docs before train:", len(mdl.docs))
print("vocabs before train:", mdl.num_vocabs)

mdl.train(1)

print("docs after train:", len(mdl.docs))
print("vocabs after train:", mdl.num_vocabs)

docs before train: 29473
vocabs before train: 0
docs after train: 29473
vocabs after train: 29416


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)


In [5]:
print("Training LDA model...")
for i in range(0, 100, 10):
    mdl.train(10)
    print(f"Iteration: {i+10}\tLog-likelihood per word: {mdl.ll_per_word:.4f}")


Training LDA model...
Iteration: 10	Log-likelihood per word: -8.7005
Iteration: 20	Log-likelihood per word: -8.1974
Iteration: 30	Log-likelihood per word: -8.0526
Iteration: 40	Log-likelihood per word: -7.9811
Iteration: 50	Log-likelihood per word: -7.9379
Iteration: 60	Log-likelihood per word: -7.9071
Iteration: 70	Log-likelihood per word: -7.8852
Iteration: 80	Log-likelihood per word: -7.8715
Iteration: 90	Log-likelihood per word: -7.8609
Iteration: 100	Log-likelihood per word: -7.8524


In [6]:
doc_topic_dist = np.array([doc.get_topic_dist() for doc in mdl.docs])

print(f"LDA fitted with {n_topics} topics")
print("Avg max topic weight:", doc_topic_dist.max(axis=1).mean())
print(f"Final log-likelihood per word: {mdl.ll_per_word:.4f}")

LDA fitted with 30 topics
Avg max topic weight: 0.36847568
Final log-likelihood per word: -7.8524


## View topics

In [7]:
topic_summaries = {}

for topic_idx in range(mdl.k):
    top_words = [w for w, _ in mdl.get_topic_words(topic_idx, top_n=15)]
    topic_summaries[topic_idx] = top_words

    print(f"Topic {topic_idx:02d}: " + ", ".join(top_words))

Topic 00: cache, processor, core, level, cores, line, memory, error, method, cache_memory, cache_line, main, store, hierarchy, processor_core
Topic 01: image, pixel, color, pixels, image_data, method, images, region, value, values, image_processing, resolution, depth, input_image, generating
Topic 02: module, channel, layer, frequency, audio, domain, filter, modules, channels, phase, method, high, multimedia, peripheral, processing_module
Topic 03: power, state, component, mode, controller, processor, components, temperature, voltage, level, management, heat, method, handling, supply
Topic 04: value, quantization, values, parameter, quantized, method, encoding, transform, encoded, compressed, compression, coefficients, decoding, coefficient, coding
Topic 05: memory, address, access, buffer, read, command, write, request, controller, storage, page, data, method, operation, stored
Topic 06: display, gpu, graphics, video, rendering, content, frame, tile, method, buffer, graphics_processin

In [ ]:
from stability import compute_topic_stability

k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0, 1, 2, 3, 4]

results_df = compute_topic_stability(
    tokenized_docs=df["tokens"].tolist(),
    k_values=k_values,
    seeds=seeds,
    iterations=100,
    vocab_min_df=10,
    min_df=10,
    rm_top=0
)

print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))

Fixed comparison vocab size: 29474

===== k = 16 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parall

Omega (theta-side):      0.523
Omega (words-side):      0.584
Omega (avg):             0.553
Omega SE (repo-style):   0.0293
Avg Max Topic Cosine:    0.380
Avg Mean Topic Cosine:   0.116

===== k = 18 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parall

Omega (theta-side):      0.474
Omega (words-side):      0.510
Omega (avg):             0.492
Omega SE (repo-style):   0.0168
Avg Max Topic Cosine:    0.427
Avg Mean Topic Cosine:   0.093

===== k = 20 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parall

Omega (theta-side):      0.521
Omega (words-side):      0.591
Omega (avg):             0.556
Omega SE (repo-style):   0.0189
Avg Max Topic Cosine:    0.469
Avg Mean Topic Cosine:   0.081

===== k = 22 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
